# Smartphone Price-Range Classification & Feature Ranking

TCS iON RIO 125 internship project: “Rank Features of a Smartphone.” Uses a public smartphone specs dataset (battery, RAM, screen, connectivity, etc.) to (1) rank which features matter most to price, and (2) train and compare classifiers that predict a phone's price range (budget/mid/premium/flagship) from its specs.

**Data:** `train.csv` / `test.csv` — place both in a `data/` folder next to this notebook before running (see the [original Kaggle Mobile Price Classification dataset](https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification) if you need to re-download them).

**Full write-up:** see `INDUSTRY_REPORT_RIO.docx` in this folder.

## 1. Imports

In [ ]:
import plotly.offline as pyo
import numpy as np 
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pandas as pd
warnings.filterwarnings("ignore")

## 2. Load the data
*(paths fixed to be relative — originally hardcoded to a local machine)*

In [ ]:
df = pd.read_csv("data/test.csv")
df = pd.read_csv("data/train.csv")  # train.csv is the one actually used below

In [ ]:
df = pd.DataFrame(df)
df.head(11)

## 3. Explore the data

In [ ]:
#data frame discription
df.describe()

In [ ]:
#null check
df.isnull()

Distribution of each categorical/low-cardinality feature:

In [ ]:
#CountPlot for various columns
for i in df:
    if(df[i].nunique()<30):
        sns.countplot(x=df[i])
        plt.show()

Distribution of each continuous feature:

In [ ]:
#Distplot for various columns
plt.figure(figsize = (30,10))
plt.subplot(331)
sns.distplot(df['battery_power'])
plt.subplot(332)
sns.distplot(df['clock_speed'])
plt.subplot(333)
sns.distplot(df['int_memory'])
plt.subplot(334)
sns.distplot(df['m_dep'])
plt.subplot(335)
sns.distplot(df['mobile_wt'])
plt.subplot(336)
sns.distplot(df['px_height'])
plt.subplot(337)
sns.distplot(df['px_width'])
plt.subplot(338)
sns.distplot(df['ram'])
plt.subplot(339)
sns.distplot(df['talk_time'])
plt.show()

## 4. Readable labels for binary features
Convert 0/1 flags into Yes/No labels and visualize the split for each.

In [ ]:
#bluetooth devices
df["is_bluetooth"]=''
for i in range(len(df)):
    if df['blue'][i]==0:
        df['is_bluetooth'][i]='No'
    else:
        df['is_bluetooth'][i]='Yes'
px.pie(data_frame = df, names = 'is_bluetooth', title = 'Percentage of devices having bluetooth', hole= 0.2)

In [ ]:
#dual sim
df["is_DualSim"]=''
for i in range(len(df)):
    if df['dual_sim'][i]==0:
        df['is_DualSim'][i]='No'
    else:
        df['is_DualSim'][i]='Yes'
px.pie(data_frame = df, names = 'is_DualSim', title = 'Percentage of devices having dual sim', hole= 0.2)


In [ ]:
#4G
df["is_4G"]=''
for i in range(len(df)):
    if df['four_g'][i]==0:
        df['is_4G'][i]='No'
    else:
        df['is_4G'][i]='Yes'
px.pie(data_frame = df, names = 'is_4G', title = 'Percentage of devices having 4G connection', hole= 0.2)

In [ ]:
#3G
df["is_3G"]=''
for i in range(len(df)):
    if df['three_g'][i]==0:
        df['is_3G'][i]='No'
    else:
        df['is_3G'][i]='Yes'
px.pie(data_frame = df, names = 'is_3G', title = 'Percentage of devices having 3G connection', hole= 0.2)

In [ ]:
#Touch Screen
df["is_touchscreen"]=''
for i in range(len(df)):
    if df['touch_screen'][i]==0:
        df['is_touchscreen'][i]='No'
    else:
        df['is_touchscreen'][i]='Yes'
px.pie(data_frame = df, names = 'is_touchscreen', title = 'Percentage of devices having touch screen', hole= 0.2)

In [ ]:
#Wifi
df["is_wifi"]=''
for i in range(len(df)):
    if df['wifi'][i]==0:
        df['is_wifi'][i]='No'
    else:
        df['is_wifi'][i]='Yes'
px.pie(data_frame = df, names = 'is_wifi', title = 'Percentage of devices having Wifi', hole= 0.2)


In [ ]:
#Processors
df["cores"]=''
for i in range(len(df)):
    if df['n_cores'][i]==1:
        df['cores'][i]='single core'
    elif df['n_cores'][i]==2:
        df['cores'][i]='dual core'
    elif df['n_cores'][i]==3:
        df['cores'][i]='triple core'
    elif df['n_cores'][i]==4:
        df['cores'][i]='quad core'
    elif df['n_cores'][i]==5:
        df['cores'][i]='penta core'
    elif df['n_cores'][i]==6:
        df['cores'][i]='hexa core'
    elif df['n_cores'][i]==7:
        df['cores'][i]='hepta core'
    else:
        df['cores'][i]='octa core'
px.pie(data_frame = df, names = 'cores', title = 'Percentage of devices having different types of cores', hole= 0.2)

## 5. Reassemble feature set

In [ ]:
#Training set
df1 = df.loc[:,['battery_power','blue','dual_sim','fc','four_g','int_memory','m_dep','mobile_wt','n_cores','pc','px_height','px_width','ram','sc_h','sc_w','talk_time','three_g','wifi','price_range']]
df1

In [ ]:
print(df.columns)

In [ ]:
df2 = df.loc[:,['touch_screen','clock_speed']]
df2

In [ ]:
df3 = pd.concat([df1, df2])
df3

## 6. Rank features by contribution to price
The core deliverable of the RIO 125 brief: rank phones by `price_range`, then rank every individual feature the same way to see which specs move together with price.

In [ ]:
df["rank_by_price"] = df["price_range"].rank()
dt1 = df
dt1.head()

In [ ]:
dt1["rank_by_price"] = dt1["rank_by_price"].sort_values()
dt1

Sort by price rank:

In [ ]:
dt1.sort_values(by=["rank_by_price"])

Re-rank against the full training data, this time ranking **every** feature (not just price), choosing ascending vs. descending per feature depending on whether higher is better (e.g. more RAM) or lower is better (e.g. less weight):

In [ ]:
dt2 = pd.read_csv("data/train.csv")
RankedDataset1 = dt2.rank()
RankedDataset1.sort_values(by="price_range")

In [ ]:
# because not all features are good when values are high and not all features are good when values are low
# It depends on each and every feature
b = dt2
b["rank_by_price"] = b["price_range"].rank()
b["rank_by_battery"] = b["battery_power"].rank(ascending=False)
b["rank_by_blueooth"] = b["blue"].rank(ascending=False)
b["rank_by_clockspeed"] = b["clock_speed"].rank(ascending=False)
b["rank_by_DualSIM"] = b["dual_sim"].rank(ascending=False)
b["rank_by_fc"] = b["fc"].rank(ascending=False)
b["rank_by_4G"] = b["four_g"].rank(ascending=False)
b["rank_by_InternalMemory"] = b["int_memory"].rank(ascending=False)
b["rank_by_mdep"] = b["m_dep"].rank(ascending=False)
b["rank_by_weight"] = b["mobile_wt"].rank(ascending=True)
b["rank_by_ncores"] = b["n_cores"].rank(ascending=False)
b["rank_by_pc"] = b["pc"].rank(ascending=False)
b["rank_by_height"] = b["px_height"].rank(ascending=False)
b["rank_by_width"] = b["px_width"].rank(ascending=False)
b["rank_by_ram"] = b["ram"].rank(ascending=False)
b["rank_by_sch"] = b["sc_h"].rank(ascending=False)
b["rank_by_scw"] = b["sc_w"].rank(ascending=False)
b["rank_by_talktime"] = b["talk_time"].rank(ascending=False)
b["rank_by_3G"] = b["three_g"].rank(ascending=False)
b["rank_by_touchscreen"] = b["touch_screen"].rank(ascending=False)
b["rank_by_wifi"] = b["wifi"].rank(ascending=False)
b.head()

Isolated per-feature rank columns — this table is the feature-ranking deliverable:

In [ ]:
RankedDataset2 = b.iloc[:,21:]
RankedDataset2

## 7. Prepare data for classification
Drop any non-numeric columns, split features/target, scale, and train/test split.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
# Assuming 'price_range' is the target column and you are excluding it from x
# Step 1: Check for any non-numeric columns
non_numeric_columns = df.select_dtypes(exclude=["number"]).columns

# If you have categorical data, you need to convert it to numeric.
# For simplicity here we drop non-numeric columns.
df_numeric = df.drop(columns=non_numeric_columns)

# Step 2: Define x and y
x = df_numeric.iloc[:, :-1].values  # All columns except the last one (features)
y = df_numeric.iloc[:, -1].values   # The last column (target)

# Step 3: Feature scaling (standardization)
sc_x = StandardScaler()
x = sc_x.fit_transform(x)

# Step 4: Train-test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=0)

print("Training and testing data prepared successfully.")

## 8. Baseline model: Logistic Regression

In [ ]:
# Re-load and bin price_range into 4 classes for classification
df = pd.read_csv("data/train.csv")
df['price_range'] = pd.cut(df['price_range'], bins=4, labels=[0, 1, 2, 3])

X = df.drop('price_range', axis=1).values
y_log = df['price_range'].values
print("Unique values in y:", set(y_log))

X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.25, random_state=0)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

log = LogisticRegression(max_iter=1000)
log.fit(X_train, y_train_log)
print("Training score of Logistic Regression is: {:.2f}%".format(log.score(X_train, y_train_log) * 100))

y_predlog = log.predict(X_test)
acrr = accuracy_score(y_test_log, y_predlog) * 100
print("Accuracy of Logistic Regression Classifier is: {:.2f}%".format(acrr))
print("Confusion matrix Logistic Regression Classifier is: \n{}".format(confusion_matrix(y_test_log, y_predlog)))
print("{}".format(classification_report(y_test_log, y_predlog)))

## 9. Compare additional classifiers
Using the `x_train`/`x_test`/`y_train`/`y_test` split from Section 7 (continuous `price_range`, not binned).

**Gaussian Naive Bayes**

In [ ]:
#Gussain NB
from sklearn.naive_bayes import GaussianNB
nb=GaussianNB()
nb.fit(x_train,y_train)
print("Training score of GaussianNB is {}".format(nb.score(x_train,y_train)*100))
y_pred=nb.predict(x_test)
ac_nb=accuracy_score(y_test,y_pred)*100
print("Accuracy of Naive Bayes Classifier is: {}".format(ac_nb))
print("Confusion matrix of Naive Bayes Classifier is:{}".format(confusion_matrix(y_test,y_pred)))
print("{}".format(classification_report(y_test,y_pred)))

**Support Vector Machine**

In [ ]:
#SVM
from sklearn.svm import SVC
svm=SVC(kernel='rbf')
svm.fit(x_train,y_train)
print("Training score of SVM is: {}".format(svm.score(x_train,y_train)*100))
y_pred_svm=svm.predict(x_test)
ac_svm=accuracy_score(y_test,y_pred_svm)*100
print("Accuracy of SVM is: {}".format(ac_svm))
print("Confusion matrix of SVM is: {}".format(confusion_matrix(y_test,y_pred_svm)))
print("{}".format(classification_report(y_test,y_pred_svm)))

**Decision Tree**

In [ ]:
#Decision Tree
from sklearn.tree import DecisionTreeClassifier
DT=DecisionTreeClassifier(criterion = "entropy")
DT.fit(x_train,y_train)
print("Training score of DecisionTreeClassifier is: {}".format(DT.score(x_train,y_train)*100))
y_pred_DT=DT.predict(x_test)
ac_DT=accuracy_score(y_test,y_pred_DT)*100
print("Accuracy of Decision Tree Classifier is: {}".format(ac_DT))
print("Confusion matrix of Decision Tree Classifier is: {}".format(confusion_matrix(y_test,y_pred_DT)))
print("{}".format(classification_report(y_test,y_pred_DT)))

**Random Forest**

In [ ]:
#Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
RF=RandomForestClassifier(n_estimators=300)
RF.fit(x_train,y_train)
print("Training score of Random Forest Classifier is: {}".format(RF.score(x_train,y_train)*100))
y_pred_RF=RF.predict(x_test)
ac_RF=accuracy_score(y_test,y_pred_RF)*100
print("Accuracy of Random Forest Classifier is: {}".format(ac_RF))
print("Confusion matrix of Random Forest Classifier is: {}".format(confusion_matrix(y_test,y_pred_RF)))
print("{}".format(classification_report(y_test,y_pred_RF)))

## 10. Compare accuracy across models

In [ ]:
#Classifiers
classifiers=["LogisticRegression","GaussianNB","SVM","DecisionTreeClassifier","RandomForestClassifier"]
accuracy_=[acrr,ac_nb,ac_svm,ac_DT,ac_RF]
df_ac=pd.DataFrame({'model':classifiers,"accuracy":accuracy_})
px.histogram(data_frame=df_ac,x="model",y="accuracy")

## 11. Tune Random Forest (regressor variant): number of estimators

In [ ]:
#Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
train_accuracy=[]
test_accuracy=[]
for i in range(100,600,100):
    c=RandomForestRegressor(n_estimators=i)
    c.fit(x_train,y_train)
    train_accuracy.append(c.score(x_train,y_train))
    test_accuracy.append(c.score(x_test,y_test))
frame=pd.DataFrame({"n_neighbors":range(100,600,100),"train_accuracy":train_accuracy,"test_accuracy":test_accuracy})
frame
plt.figure(figsize=(10,7))
plt.plot(range(100,600,100),frame["train_accuracy"],label="Train")
plt.plot(range(100,600,100),frame["test_accuracy"],label="Test")
plt.title("Optimal Number of Estimators")
plt.xlabel("Estimators")
plt.ylabel("Accuracy")
plt.legend()
plt.show()
print(frame)

## 12. Cross-validate all classifiers (10-fold)
Single train/test splits can be noisy — cross-validation gives a more reliable comparison.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

kfold = KFold(n_splits = 10)
xyz = []
accuracy = []

classifiers=["SVM","Logistic Regression","Decision Tree","Random Forest","GaussianNB"]
models=[SVC(kernel='rbf'),LogisticRegression(),DecisionTreeClassifier(),RandomForestClassifier(n_estimators=300,random_state=0),GaussianNB()]

for i in models:
    model = i
    cv_result=cross_val_score(model,x_train,y_train,cv=kfold,scoring="accuracy")
    cv_result = cv_result
    xyz.append(cv_result.mean())
    accuracy.append(cv_result)
    
cv_models_datafeame= pd.DataFrame(xyz, index = classifiers)
cv_models_datafeame.columns = ['CV Mean']
cv_models_datafeame
cv_models_datafeame.sort_values(['CV Mean'], ascending =[0])

box = pd.DataFrame(accuracy, index = [classifiers])
boxT = box.T
plt.figure(figsize = (10,7))
ax = sns.boxplot(data = boxT, orient = "h", palette = "Set2", width = 0.6)
ax.set_yticklabels(classifiers)
ax.set_title('Cross Validation accuracy with different classifiers')
ax.set_xlabel('Accuracy')

plt.show()


## Conclusion
Random Forest and SVM are the strongest performers in cross-validation, with Logistic Regression as a solid, more interpretable baseline. The feature-ranking step (Section 6) shows RAM, battery power, and pixel resolution as the specs most consistently associated with higher price range — consistent with the model's learned feature importances. See `INDUSTRY_REPORT_RIO.docx` for the full discussion.